# Data Exploration — MovieLens 25M Recommender System

This notebook performs exploratory data analysis (EDA) on the MovieLens 25M dataset.

The main objectives are to:

- Inspect the structure and quality of the datasets
- Analyze rating distributions
- Examine user and movie activity
- Explore movie genres
- Identify potential characteristics of the recommendation problem
- Measure dataset sparsity

The results of this analysis are used to guide the subsequent recommendation-system and feature-engineering steps.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Load Dataset

The MovieLens 25M dataset contains movie metadata and user ratings.

The two main files used in this project are:

- `movies.csv`: movie IDs, titles, and genres
- `ratings.csv`: user–movie ratings

Only the columns required for the recommendation system are analyzed in this notebook.

In [ ]:
movies = pd.read_csv("../data/raw/ml-25m/movies.csv")
ratings = pd.read_csv("../data/raw/ml-25m/ratings.csv")
movies.head()

In [ ]:
print("Movies shape: ", movies.shape)
print("Ratings shape: ", ratings.shape)

## Data Quality and Basic Statistics

Before analyzing user behavior and movie popularity, the datasets are checked for missing values, duplicate records, and invalid relationships between users, movies, and ratings.

In [ ]:
print("Missing values in Movies: ")
print(movies.isnull().sum())

print("Missing values in Ratings: ")
print(ratings.isnull().sum())

### Basic rating statistics

In [ ]:
ratings.describe()

In [ ]:
movie_rating_counts = (ratings.groupby("movieId").size().sort_values(ascending=False))
movie_rating_counts.head(10)

In [ ]:
ratings_with_movies = ratings.merge(movies, on= "movieId", how= "left")
ratings_with_movies.head()

In [ ]:
print("Duplicate ratings :", ratings.duplicated().sum())
print("Missing values: ", ratings_with_movies.isnull().sum())

## Rating Distribution

Understanding the distribution of user ratings helps identify rating preferences and the overall behavior of the rating data.

In [ ]:
rating_distribution = (ratings["rating"].value_counts().sort_index())
rating_distribution

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x=rating_distribution.index, y=rating_distribution.values)
plt.title("Rating distribution")
plt.xlabel("Ratings")
plt.ylabel("Number of Ratings")
plt.tight_layout()
plt.show()

## User Activity

The number of ratings submitted by each user is examined to understand the distribution of user activity.

This is particularly relevant for collaborative filtering because users contribute different amounts of interaction data.

In [ ]:
rating_per_user = ratings.groupby("userId").size().sort_values(ascending=False)
rating_per_user.describe()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(rating_per_user, bins=50)
plt.title("Rating per user")
plt.xlabel("Number of Ratings")
plt.ylabel("Number of users")
plt.tight_layout()
plt.show()

## Movie Popularity

The number of ratings received by each movie is analyzed as a simple measure of movie popularity.

A highly skewed distribution may indicate a popularity bias in the interaction data.

In [ ]:
rating_per_movie = ratings.groupby("movieId").size().sort_values(ascending=False)
rating_per_movie.describe()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(rating_per_movie, bins=50)
plt.title("Rating per movie")
plt.xlabel("Number of Ratings")
plt.ylabel("Number of movies")
plt.tight_layout()
plt.show()

### Most rated movies

In [ ]:
top_rated_movies = (ratings.groupby("movieId").size().sort_values(ascending=False).head(10).reset_index(name="rating_count"))
top_rated_movies = top_rated_movies.merge(movies, on="movieId", how="left")
top_rated_movies

## Movie Popularity

The number of ratings received by each movie is analyzed as a simple measure of movie popularity.

A highly skewed distribution may indicate a popularity bias in the interaction data.

In [ ]:
unique_genres = movies['genres'].str.split("|").explode().unique()
print(unique_genres)
print('Number of genres:',len(unique_genres))

##  Genre Distribution

Movie genres are explored to understand the composition of the movie catalog and provide context for the content-based recommendation component.

In [ ]:
genre_counts = movies['genres'].str.split("|").explode().value_counts()
genre_counts

In [ ]:
plt.figure(figsize=(10, 8))
sns.barplot(x=genre_counts.values, y=genre_counts.index, orient="h")
plt.title("Number of movies by genre")
plt.xlabel("Number of movies")
plt.ylabel("Genres")
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.show()

In [ ]:
movie_features = movies.copy()
movie_features.head(5)

In [ ]:
genre_features = (movie_features['genres'].str.get_dummies(sep="|"))
genre_features.head()

In [ ]:
movie_features = pd.concat([movie_features[["movieId", "title"]], genre_features], axis=1)
movie_features.head()

In [ ]:
print("Movies features shape: ", movie_features.shape)
print("Missing values: ", movie_features.isnull().sum().sum())

## Dataset Summary and Sparsity

The overall size of the recommendation problem is summarized using the number of users, movies, and ratings.

Sparsity measures how large the user–movie interaction space is compared with the number of observed ratings. High sparsity is a common characteristic of collaborative filtering datasets.

In [ ]:
n_users = ratings["userId"].nunique()
n_movies = ratings["movieId"].nunique()
n_ratings = len(ratings)

total_possible_ratings = n_users * n_movies

sparsity = 1 - (n_ratings / total_possible_ratings)

print(f"Users: {n_users:,}")
print(f"Movies: {n_movies:,}")
print(f"Ratings: {n_ratings:,}")
print(f"Sparsity: {sparsity:.2%}")

In [ ]:
print("Unique movie IDs in movies:", movies["movieId"].nunique())

print("Duplicate movie IDs in movies:", movies["movieId"].duplicated().sum())

print("Duplicate user-movie pairs in ratings:", ratings.duplicated(subset=["userId", "movieId"]).sum())

print("Ratings with missing movies:", (~ratings["movieId"].isin(movies["movieId"])).sum())

In [ ]:
users = pd.DataFrame({"user_id": ratings["userId"].unique()})
movies_sql = movies[["movieId", "title"]].copy()
ratings_sql = ratings[["userId", "movieId", "rating"]].copy()
ratings_sql.head()

In [ ]:
genres = (movies["genres"].str.split("|").explode().drop_duplicates().sort_values().reset_index(drop=True))
genres = pd.DataFrame({"genre_id" : range(1, len(genres) + 1), "genre_name": genres})
genres

In [ ]:
movie_genres = (movies[["movieId", "genres"]].assign(genre_name= movies["genres"].str.split("|")).explode("genre_name").drop(columns="genres"))
movie_genres = movie_genres.merge(genres, on="genre_name", how="left")
movie_genres = movie_genres[["movieId", "genre_id"]]
movie_genres



In [ ]:
dataframes = {
    "users": users,
    "movies_sql": movies_sql,
    "ratings_sql": ratings_sql,
    "genres": genres,
    "movie_genres": movie_genres
}

for name, dataframe in dataframes.items():
    print(f"{name}: {dataframe.shape}")
for name, dataframe in dataframes.items():
    print(f"\n{name}")
    print(dataframe.dtypes)
    print("-" * 40)
print("Duplicate users:", users["user_id"].duplicated().sum())

print("Duplicate movies:", movies_sql["movieId"].duplicated().sum())

print(
    "Duplicate movie-genre pairs:",
    movie_genres.duplicated(subset=["movieId", "genre_id"]).sum()
)
print("Missing genre IDs:", movie_genres["genre_id"].isna().sum())

##  Key Findings

The exploratory analysis highlights several characteristics of the MovieLens 25M dataset:

- The user–movie interaction matrix is highly sparse.
- User activity is uneven, with some users providing substantially more ratings than others.
- Movie popularity is also highly uneven, with a relatively small number of movies receiving many ratings.
- Rating values are concentrated around the middle-to-high end of the rating scale.
- Movie genres provide useful metadata for content-based recommendation.

These findings motivate the use of both collaborative and content-based approaches. The next steps of the project focus on feature engineering, model development, and evaluation.